# 02 V2 — WDC Download (All Types, English-Speaking Web)

Downloads Web Data Commons class-specific N-Quads for **all** target schema.org types
(not just LocalBusiness + Product as in v1).

WDC gives us real-world URLs of pages that already have schema markup — we use these
as a URL seed for live-fetching in notebook 03_v2. We don't use the existing JSON-LD
as labels; Gemini Flash generates fresh, richer labels from HTML + screenshot.

## Types covered
LocalBusiness, Product, Organization, Article, NewsArticle, BlogPosting,
Event, FAQPage, Recipe, Person, WebSite

## Output
`data/raw/wdc_candidate_urls.jsonl` — `{url, schema_type, property_count, source}`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

WDC_DIR  = Path('../data/raw/wdc')
OUT_PATH = Path('../data/raw/wdc_candidate_urls.jsonl')
WDC_DIR.mkdir(parents=True, exist_ok=True)

from src.wdc import WDC_PART_COUNTS, download_wdc_subset, load_and_filter_wdc

print('Available WDC types:', list(WDC_PART_COUNTS.keys()))

## 1. Configure Target Types and Quotas

Match to `config/type_distribution.json` weights.
Quotas are URL counts to collect per type — more than we need so 03_v2 can filter.

In [ ]:
# URLs to collect per type (generous — 03_v2 will filter/sample down)
# WDC types that map to our target distribution
TYPE_QUOTAS = {
    'LocalBusiness':  30_000,   # 20% target — many subtypes
    'Product':        20_000,   # 15% target
    'Article':        15_000,   # 10% — includes NewsArticle + BlogPosting
    'NewsArticle':     5_000,
    'BlogPosting':     5_000,
    'Event':          12_000,   # 8%
    'FAQPage':         8_000,   # 5%
    'Recipe':         10_000,   # 7%
    'Organization':   10_000,   # 7%
    'Person':          5_000,   # 3%
    'WebSite':         5_000,   # 3% — good for homepage schema
}

print('Type quotas (URL targets):')
total = sum(TYPE_QUOTAS.values())
for t, q in TYPE_QUOTAS.items():
    print(f'  {t:20s} {q:6,}  ({q/total*100:.0f}%)')
print(f'  {"TOTAL":20s} {total:6,}')

## 2. Download WDC Files

Downloads class-specific N-Quads (.nq.gz) files. Large files (1-10 GB compressed).
Already-downloaded files are skipped automatically.

In [ ]:
downloaded = {}  # schema_type → Path

for schema_type, quota in TYPE_QUOTAS.items():
    # Check if already downloaded
    existing = list(WDC_DIR.glob(f'*{schema_type}*.gz'))
    if existing:
        print(f'  {schema_type}: already downloaded ({existing[0].name})')
        downloaded[schema_type] = existing[0]
        continue

    if schema_type not in WDC_PART_COUNTS:
        print(f'  {schema_type}: not in WDC, skipping')
        continue

    try:
        path = download_wdc_subset(schema_type, str(WDC_DIR))
        downloaded[schema_type] = path
        print(f'  {schema_type}: downloaded → {path}')
    except Exception as e:
        print(f'  {schema_type}: FAILED — {e}')

print(f'\nDownloaded {len(downloaded)}/{len(TYPE_QUOTAS)} types')

## 3. Parse and Filter

Extract URLs from each N-Quads file. Filter for:
- English-language domains (heuristic: `.com`, `.co.uk`, `.org`, `.net`, `.ie`, `.ca`, `.com.au`, `.co.nz`, `.us`)
- Minimum property count (quality signal)
- Valid absolute URLs

In [ ]:
from urllib.parse import urlparse

ENGLISH_TLDS = {
    'com', 'org', 'net', 'ie', 'us', 'biz', 'info',
    'co.uk', 'org.uk', 'me.uk',         # UK
    'ca',                                # Canada
    'com.au', 'net.au', 'org.au',        # Australia
    'co.nz', 'net.nz', 'org.nz',         # New Zealand
}

def is_english_tld(url: str) -> bool:
    try:
        host = urlparse(url).netloc.lower().lstrip('www.')
        for tld in ENGLISH_TLDS:
            if host.endswith('.' + tld) or host == tld:
                return True
    except Exception:
        pass
    return False

all_urls = []
type_counts = Counter()

for schema_type, filepath in downloaded.items():
    if filepath is None:
        continue
    quota = TYPE_QUOTAS.get(schema_type, 5_000)

    records = load_and_filter_wdc(
        filepath=str(filepath),
        schema_type=schema_type,
        tld_filter=None,          # we filter ourselves below
        min_properties=3,         # low threshold — 03_v2 does quality filtering
        max_records=quota * 3,    # pull extra so filtering doesn't starve us
    )

    kept = 0
    for rec in records:
        url = rec.get('source_url', '')
        if not url.startswith('http'):
            continue
        if not is_english_tld(url):
            continue
        all_urls.append({
            'url':            url,
            'schema_type':    schema_type,
            'property_count': int(rec.get('property_count', 0)),
            'source':         'wdc',
        })
        kept += 1
        if kept >= quota:
            break
    type_counts[schema_type] = kept
    print(f'  {schema_type:20s} {kept:6,} URLs (English-TLD filtered)')

print(f'\nTotal WDC candidate URLs: {len(all_urls):,}')

In [ ]:
# Deduplicate by URL
seen = set()
deduped = []
for r in all_urls:
    if r['url'] not in seen:
        seen.add(r['url'])
        deduped.append(r)

print(f'After dedup: {len(deduped):,} unique URLs')

# Save
with open(OUT_PATH, 'w') as f:
    for r in deduped:
        f.write(json.dumps(r) + '\n')

size_mb = OUT_PATH.stat().st_size / 1e6
print(f'Saved → {OUT_PATH} ({size_mb:.1f} MB)')

## 4. Merge with CC Candidates

Combine WDC URLs with the CC query results from 01_v2 into a single merged candidate pool.

In [ ]:
CC_PATH     = Path('../data/raw/cc_candidate_urls.jsonl')
MERGED_PATH = Path('../data/raw/all_candidate_urls.jsonl')

merged = list(deduped)  # start with WDC
wdc_urls = seen.copy()

if CC_PATH.exists():
    cc_added = 0
    with open(CC_PATH) as f:
        for line in f:
            r = json.loads(line)
            # CC records use 'probable_type', normalise to 'schema_type'
            r.setdefault('schema_type', r.pop('probable_type', 'LocalBusiness'))
            if r['url'] not in wdc_urls:
                merged.append(r)
                wdc_urls.add(r['url'])
                cc_added += 1
    print(f'Added {cc_added:,} CC URLs (deduped against WDC)')
else:
    print('No CC candidates found — run 01_v2_crawl_query.ipynb first')

with open(MERGED_PATH, 'w') as f:
    for r in merged:
        f.write(json.dumps(r) + '\n')

print(f'\nMerged pool: {len(merged):,} URLs → {MERGED_PATH}')
print('\nType breakdown:')
for t, n in Counter(r['schema_type'] for r in merged).most_common():
    print(f'  {t:25s} {n:6,}')
print('\nNext: 03_v2_html_extraction.ipynb')